# 04 - Grade Band Specifications
**Module:** `src/prompts/grade_specs.py`

## What this module does?
Encodes all research-grounded grade band rules as Python data.
These rules are injected directly into every prompt. They are
what makes a K-2 lesson fundamentally different from a 9-12 lesson.

## Research grounding?
- **Sweller (1988)** — Cognitive Load Theory: working memory limits
  differ by developmental stage. Younger learners need more scaffolding.
- **Chall (1983)** — Stages of Reading Development: language complexity
  expectations by grade band.
- **CCSS ELA Standards** — Speaking & Listening anchor standards
  by grade band cluster.
- **Bruner (1990)** — Narrative types appropriate for different ages.

## What we'll explore?
1. The structure of a single grade spec
2. Comparing vocab ceilings across grade bands
3. Comparing scaffolding levels
4. Comparing narrative types
5. Comparing forbidden elements across grade bands
6. The formatted spec block that gets injected into prompts
7. Error handling


In [2]:
import sys
sys.path.insert(0, '..')

from src.prompts.grade_specs import (
    GRADE_SPECS,
    get_spec,
    get_vocab_ceiling,
    format_spec_for_prompt,
)

print(f"Grade bands loaded: {list(GRADE_SPECS.keys())}")

Grade bands loaded: ['K-2', '3-5', '6-8', '9-12']


## Part 1: Anatomy of a Single Grade Spec
Let's inspect the K-2 spec in full...every field and what it means.

In [6]:
# Inspect the full 9-12 spec
spec = get_spec("9-12")

print("=== 9-12 GRADE SPEC ===\n")
for key, value in spec.items():
    if isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}:")
        print(f"  {value}")
    print()

=== 9-12 GRADE SPEC ===

cognitive_load_rule:
  Full complexity is appropriate. Challenge should come from the intellectual depth of the content, not from confusing instructions. Keep task framing clear even when the content is sophisticated. Chunk multi-part tasks explicitly.

vocab_ceiling:
  150

vocab_ceiling_note:
  Maximum 150 words per speaking prompt. Academic and domain-specific vocabulary is expected. Rhetorical, analytical, and discipline-specific terms are appropriate without inline definition.

narrative_types:
  - Real-world policy and civic issues
  - Philosophical and ethical questions
  - Literary and rhetorical analysis
  - Professional simulations (debates, interviews, presentations)
  - Historical primary source analysis
  - Scientific controversy and evidence evaluation

speaking_task_length:
  5-8 sentences or a structured short speech (1-2 minutes). Students should demonstrate claim, evidence, reasoning, counterargument acknowledgment, and audience awareness.

sc

## Part 2: Comparing Vocab Ceilings Across Grade Bands
The vocabulary ceiling directly implements Sweller's (1988) working
memory principle. Younger learners have less capacity for processing
long instructions while simultaneously producing spoken language.

In [11]:
# Vocab ceiling comparison
print("Vocabulary ceiling per grade band:\n")
print(f"  {'Grade Band':<10} {'Max Words':<12} {'Research Basis'}")
print(f"  {'-'*10:<10} {'-'*10:<12} {'-'*48}")

descriptions = {
    "K-2":  "Sweller (1988) — very limited working memory",
    "3-5":  "Chall (1983) Stage 2-3 — reading to learn begins",
    "6-8":  "Full CLT — all dimensions can be novel",
    "9-12": "Chall (1983) Stage 4-5 — multiple viewpoints",
}

for band in ["K-2", "3-5", "6-8", "9-12"]:
    ceiling = get_vocab_ceiling(band)
    print(f"  {band:<10} {ceiling:<12} {descriptions[band]}")

Vocabulary ceiling per grade band:

  Grade Band Max Words    Research Basis
  ---------- ----------   ------------------------------------------------
  K-2        30           Sweller (1988) — very limited working memory
  3-5        60           Chall (1983) Stage 2-3 — reading to learn begins
  6-8        100          Full CLT — all dimensions can be novel
  9-12       150          Chall (1983) Stage 4-5 — multiple viewpoints


## Part 3: Comparing Scaffold Requirements
Scaffolds (sentence starters) are required for younger grades
and removed for older grades. This mirrors Vygotsky's Zone of
Proximal Development and the gradual release of responsibility model.

In [15]:
# Scaffold comparison across grade bands
print("Scaffold requirements across grade bands:\n")

for band in ["K-2", "3-5", "6-8", "9-12"]:
    spec = get_spec(band)
    required = spec["scaffold_required"]
    icon = "✅ Required" if required else "❌ Not required"
    print(f"  {band}: {icon}")
    print(f"    → {spec['scaffold_note']}")
    print()

Scaffold requirements across grade bands:

  K-2: ✅ Required
    → ALL practice prompts must include a sentence starter. Example: 'Start with: First, the turtle...' Scaffolds reduce extraneous cognitive load so students can focus entirely on the target skill (Sweller, 1994).

  3-5: ✅ Required
    → First practice prompt must include a sentence starter. Second prompt should be independent — this gradual release mirrors Vygotsky's Zone of Proximal Development.

  6-8: ❌ Not required
    → Scaffolds are optional — offer a structural framework (e.g. 'claim → reason → counterargument') but do not provide sentence starters. Students at this level benefit from constructing their own language.

  9-12: ❌ Not required
    → No sentence starters. Structural frameworks are optional and should be offered as one possible approach, not a required template. Students should develop their own rhetorical voice.



## Part 4: Narrative Types by Grade Band
Bruner (1990) argues that the type of narrative a learner can engage
with grows in complexity with age -- from concrete animal stories
to abstract philosophical questions.

In [16]:
# Narrative types comparison
print("Appropriate narrative types by grade band:\n")

for band in ["K-2", "3-5", "6-8", "9-12"]:
    spec = get_spec(band)
    print(f"  {band}:")
    for narrative in spec["narrative_types"]:
        print(f"    - {narrative}")
    print()

Appropriate narrative types by grade band:

  K-2:
    - Animal characters with clear motivations
    - Fantasy settings (forests, magical worlds, talking objects)
    - Simple cause-and-effect plots (X happened, so Y happened)
    - Familiar community settings (school, home, playground)

  3-5:
    - Adventure and mystery plots
    - Science discovery scenarios (expeditions, experiments)
    - Historical fiction (accessible, character-focused)
    - Community and social problem-solving stories
    - Animal and nature narratives with scientific grounding

  6-8:
    - Ethical dilemmas and moral decision-making
    - Current events and social issues (balanced framing required)
    - Historical turning points and counterfactual thinking
    - Science and technology ethics
    - Interpersonal conflict and perspective-taking

  9-12:
    - Real-world policy and civic issues
    - Philosophical and ethical questions
    - Literary and rhetorical analysis
    - Professional simulations (deba

## Part 5: Forbidden Elements
Each grade band has a list of things the LLM must never include.
These are the guardrails that prevent age-inappropriate content.

In [17]:
# Forbidden elements — what the LLM must avoid per grade band
print("Forbidden elements by grade band:\n")

for band in ["K-2", "3-5", "6-8", "9-12"]:
    spec = get_spec(band)
    print(f"  {band}:")
    for item in spec["forbidden_elements"]:
        print(f"    ✗ {item}")
    print()

Forbidden elements by grade band:

  K-2:
    ✗ Abstract or philosophical themes
    ✗ Multi-part instructions (more than one thing at a time)
    ✗ Vocabulary requiring prior content knowledge
    ✗ Idioms or figurative language without explanation
    ✗ Historical or current events content
    ✗ Any prompt requiring more than 2 sentences to answer

  3-5:
    ✗ Highly abstract philosophical questions without concrete grounding
    ✗ Politically charged content without clear balanced framing
    ✗ Technical vocabulary without inline definitions
    ✗ Speaking tasks requiring more than 4 sentences

  6-8:
    ✗ One-sided political content without balanced framing
    ✗ Topics that require lived experience students may not have
    ✗ Overly simplified binary choices on complex issues

  9-12:
    ✗ Overly simplified tasks that don't challenge this age group
    ✗ Sentence starters or heavy scaffolding (patronising at this level)
    ✗ Topics without any genuine intellectual complexity



## Part 6: The Formatted Spec Block
This is what actually gets injected into the LLM prompt.
Let's see what the model reads for each grade band.

In [18]:
# See the formatted prompt block for K-2
print(format_spec_for_prompt("K-2"))

GRADE BAND: K-2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

COGNITIVE LOAD RULE:
    Introduce only ONE new element per lesson — either a new theme OR a new skill, never both simultaneously. Sweller (1988): young learners have limited working memory and cannot process multiple novel inputs at once.

VOCABULARY:
    Ceiling: 30 words per speaking prompt.
    Maximum 30 words per speaking prompt. Use simple, high-frequency vocabulary only. No idioms, figurative language, or multi-clause sentences.

SPEAKING TASK LENGTH:
    1-2 sentences maximum. Students at this level cannot sustain longer spoken responses without losing the thread of their own thought.

SCAFFOLD REQUIRED: True
    ALL practice prompts must include a sentence starter. Example: 'Start with: First, the turtle...' Scaffolds reduce extraneous cognitive load so students can focus entirely on the target skill (Sweller, 1994).

APPROPRIATE NARRATIVE TYPES:
    - Animal characters with clear motivations
    - Fantasy settings (fo

In [20]:
# Compare the formatted blocks side by side — length tells a story
print("Formatted spec lengths (chars):\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    block = format_spec_for_prompt(band)
    print(f"  {band}: {len(block):,} chars")

Formatted spec lengths (chars):

  K-2: 1,981 chars
  3-5: 1,986 chars
  6-8: 2,119 chars
  9-12: 2,138 chars


In [21]:
# See the 9-12 spec — notice how different the tone is from K-2
print(format_spec_for_prompt("9-12"))

GRADE BAND: 9-12
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

COGNITIVE LOAD RULE:
    Full complexity is appropriate. Challenge should come from the intellectual depth of the content, not from confusing instructions. Keep task framing clear even when the content is sophisticated. Chunk multi-part tasks explicitly.

VOCABULARY:
    Ceiling: 150 words per speaking prompt.
    Maximum 150 words per speaking prompt. Academic and domain-specific vocabulary is expected. Rhetorical, analytical, and discipline-specific terms are appropriate without inline definition.

SPEAKING TASK LENGTH:
    5-8 sentences or a structured short speech (1-2 minutes). Students should demonstrate claim, evidence, reasoning, counterargument acknowledgment, and audience awareness.

SCAFFOLD REQUIRED: False
    No sentence starters. Structural frameworks are optional and should be offered as one possible approach, not a required template. Students should develop their own rhetorical voice.

APPROPRIATE NARRATIVE TYPE

## Part 7: Error Handling
What happens when an invalid grade band is requested?

In [22]:
# Invalid grade band
try:
    get_spec("college")
except KeyError as e:
    print("Caught KeyError ✅")
    print(e)

Caught KeyError ✅
"[grade_specs] Unknown grade band 'college'. Valid options: ['K-2', '3-5', '6-8', '9-12']"


## Summary

| What we explored | Key takeaway |
|---|---|
| Vocab ceilings | Escalate from 30 → 60 → 100 → 150 words across grade bands |
| Scaffold requirements | Required for K-2 and 3-5, removed for 6-8 and 9-12 |
| Narrative types | Concrete/fantasy for young → abstract/philosophical for older |
| Forbidden elements | Each grade band has explicit prohibitions for the LLM |
| Formatted spec block | K-2 and 9-12 specs read completely differently in tone |
| Error handling | Catches invalid grade bands and returns helpful error messages |

**Design decision this validates:**
Grade band rules are stored as data, not hardcoded into prompts.
This means changing a rule (e.g. raising the K-2 vocab ceiling from
30 to 40 words) requires editing one dict entry, not hunting through
prompt strings. This is a maintainability decision grounded in the
separation of data from logic.

**Research connection:**
The escalating complexity across grade bands directly implements
Sweller's (1988) cognitive load framework -- each band allows more
simultaneous novelty as working memory capacity increases with age.